# Serving and benchmarking an open-source LLM with vLLM (Colab)

This notebook serves a Qwen3 model with vLLM under several configurations, load tests each one,
scores the output quality on a receipt-extraction task with known answers, and builds the comparison tables and charts.

**Before you start**
1. Runtime > Change runtime type > pick the best GPU you have (T4, L4 or A100).
2. Upload `llm-serving-bench.zip` when the first code cell asks for it.
3. If Colab asks you to restart the runtime after installing vLLM, do it, then run again from the top.

Everything is written to `results/` and copied to Google Drive after each configuration, so a disconnect does not lose your numbers.

In [ ]:
import os, sys, zipfile, subprocess, json, time, signal, shutil

PROJECT = "/content/llm-serving-bench"
if not os.path.exists(PROJECT):
    from google.colab import files
    print("Upload llm-serving-bench.zip")
    up = files.upload()
    zipfile.ZipFile(list(up)[0]).extractall("/content")
os.chdir(PROJECT)
os.makedirs("results", exist_ok=True)
print("working in", os.getcwd())

def run(cmd, check=True):
    # run a command and print its output in the notebook
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-6000:] + (p.stderr[-3000:] if p.returncode else ""))
    if check and p.returncode:
        raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return p

In [ ]:
# 1. What GPU did we get? Write it down: numbers from different GPUs are not comparable.
run(["nvidia-smi"], check=False)
try:
    import torch
    print("compute capability:", torch.cuda.get_device_capability(0))
except Exception as e:
    print("torch not ready yet:", e)

In [ ]:
# 2. Install vLLM (takes a few minutes). Record the version in your README.
run([sys.executable, "-m", "pip", "install", "-q", "vllm", "httpx", "matplotlib"], check=False)
import vllm
print("vllm version:", vllm.__version__)

In [ ]:
# 3. Save results to Google Drive so a disconnect does not lose them.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/llm-serving-bench-results"
os.makedirs(DRIVE_OUT, exist_ok=True)

## Configurations to compare

Edit this list. Keep each change small so you can explain what caused a difference.

- **fp16 vs 4-bit (AWQ):** memory, speed and quality trade-off.
- **Prefix caching on vs off:** every request shares the same long system prompt, so prefix caching should lower time to first token.
- **4B vs 8B:** a bigger model in 4-bit against a smaller one in fp16.

Check that each model name exists on Hugging Face before running. T4 has no bfloat16, so `--dtype half` is used.
On an L4 or A100 you may raise `--max-model-len` and try larger models.

In [ ]:
COMMON = ["--dtype", "half", "--max-model-len", "2048", "--gpu-memory-utilization", "0.88", "--max-num-seqs", "32"]
PORT = 8000

CONFIGS = [
    dict(label="vllm_qwen3-4b_fp16_prefix-on",  model="Qwen/Qwen3-4B",     args=["--enable-prefix-caching"]),
    dict(label="vllm_qwen3-4b_fp16_prefix-off", model="Qwen/Qwen3-4B",     args=["--no-enable-prefix-caching"]),
    dict(label="vllm_qwen3-4b_awq_prefix-on",   model="Qwen/Qwen3-4B-AWQ", args=["--enable-prefix-caching"]),
    dict(label="vllm_qwen3-8b_awq_prefix-on",   model="Qwen/Qwen3-8B-AWQ", args=["--enable-prefix-caching"]),
]

LEVELS = "1,4,8,16"          # concurrent requests
REQUESTS_PER_LEVEL = 32
N_EVAL = 100                 # receipts scored for quality

In [ ]:
import httpx

def start_server(cfg):
    log = open(f"results/server_{cfg['label']}.log", "w")
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", cfg["model"], "--port", str(PORT)] + COMMON + cfg["args"]
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, preexec_fn=os.setsid)

def wait_ready(p, label, timeout=1200):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if p.poll() is not None:
            raise RuntimeError(f"server exited early, see results/server_{label}.log")
        try:
            if httpx.get(f"http://localhost:{PORT}/v1/models", timeout=2).status_code == 200:
                return time.time() - t0
        except Exception:
            pass
        time.sleep(5)
    raise TimeoutError("server did not become ready")

def stop_server(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        p.wait(timeout=60)
    except Exception:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception:
            pass
    time.sleep(8)  # let GPU memory free up before the next model

def tail(path, n=25):
    try:
        print("".join(open(path).readlines()[-n:]))
    except Exception:
        pass

## Run everything

For each configuration: start vLLM, record how long it takes to become ready (cold start), run the load test, run the quality evaluation, stop the server, and copy results to Drive.
If one configuration fails (for example, out of GPU memory), the log tail is printed and the loop moves on to the next one.

In [ ]:
for cfg in CONFIGS:
    label, model = cfg["label"], cfg["model"]
    print("=" * 30, label)
    p = start_server(cfg)
    try:
        ready_s = wait_ready(p, label)
        print(f"ready in {ready_s:.0f}s (cold start, includes weight load and CUDA graph capture)")
        json.dump({"label": label, "model": model, "cold_start_s": round(ready_s, 1), "vllm_args": COMMON + cfg["args"]},
                  open(f"results/startup_{label}.json", "w"), indent=2)
        run([sys.executable, "src/loadtest.py", "--base-url", f"http://localhost:{PORT}", "--model", model,
             "--label", label, "--levels", LEVELS, "--requests-per-level", str(REQUESTS_PER_LEVEL)])
        run([sys.executable, "src/quality_eval.py", "--base-url", f"http://localhost:{PORT}", "--model", model,
             "--label", label, "--n", str(N_EVAL)])
    except Exception as e:
        print("FAILED:", e)
        tail(f"results/server_{label}.log")
    finally:
        stop_server(p)
        for f in os.listdir("results"):
            shutil.copy(os.path.join("results", f), DRIVE_OUT)

In [ ]:
# Tables and charts
run([sys.executable, "src/summarize.py"])
from IPython.display import Markdown, Image, display
display(Markdown(open("results/summary.md").read()))
for f in ["throughput_vs_concurrency.png", "p95_latency_vs_concurrency.png", "quality_vs_speed.png"]:
    if os.path.exists(f"results/{f}"):
        display(Image(f"results/{f}"))
for f in os.listdir("results"):
    shutil.copy(os.path.join("results", f), DRIVE_OUT)
print("copied to", DRIVE_OUT)

## Mac baseline with Ollama (optional, 20 minutes)

The same scripts work against Ollama, so you get a comparison between a laptop and a GPU server.
On your Mac, in the project folder:

```bash
ollama pull qwen3:4b
OLLAMA_NUM_PARALLEL=8 ollama serve   # allow concurrent requests, otherwise they queue
python src/loadtest.py --base-url http://localhost:11434 --model qwen3:4b --label ollama_qwen3-4b_mac
python src/quality_eval.py --base-url http://localhost:11434 --model qwen3:4b --label ollama_qwen3-4b_mac
```

Upload the two resulting JSON files into this notebook's `results/` folder and rerun the summary cell.

## What to write down

- GPU model, vLLM version, model names, and the exact command-line flags (they are saved in `results/startup_*.json`).
- Which configuration won on throughput, which won on time to first token, and whether quality changed (look at the confidence intervals before claiming a difference).
- One thing that surprised you, and one limitation (for example: one GPU type, one synthetic workload, 100 evaluation examples).